In [1]:
import rasterio
import numpy as np

# Reference grid
reference = r"D:\landslide\final_data\distance_roads_tehri.tif"

# List ALL aligned rasters (10m versions only)
rasters = [
    r"D:\landslide\final_data\Elevation_10m.tif",
    r"D:\landslide\final_data\Slope_deg_10m.tif",
    r"D:\landslide\final_data\Aspect_deg_10m.tif",
    r"D:\landslide\final_data\Curvature_10m.tif",
    r"D:\landslide\final_data\TRI_10m.tif",
    r"D:\landslide\final_data\TWI_clean_10m.tif",
    r"D:\landslide\final_data\Rainfall_25yr_Mean_UTM_32644.tif",
    r"D:\landslide\final_data\distance_streams_tehri.tif",
    r"D:\landslide\final_data\distance_roads_tehri.tif",
    r"D:\landslide\final_data\distance_faults_tehri.tif",
    r"D:\landslide\final_data\NDVI_Tehri_10m.tif",
    r"D:\landslide\final_data\BSI_lite_Tehri_10m.tif",
    r"D:\landslide\final_data\Soil_Tehri_10m.tif",
    r"D:\landslide\final_data\lithology_tehri_10m.tif",
    r"D:\landslide\final_data\lulc_tehri_10m.tif",
    r"D:\landslide\final_data\Geomorphon_clean_10m.tif"
]

with rasterio.open(reference) as ref:
    meta = ref.meta.copy()
    height, width = ref.height, ref.width

meta.update(count=len(rasters), dtype='float32')

stack = np.zeros((len(rasters), height, width), dtype=np.float32)

for i, path in enumerate(rasters):
    with rasterio.open(path) as src:
        stack[i, :, :] = src.read(1)

output_stack = r"D:\landslide\landslide_raster\LSM_stack_new_10m.tif"

with rasterio.open(output_stack, "w", **meta) as dst:
    dst.write(stack)

print("Stack created successfully.")

Stack created successfully.


In [2]:
stack_path = r"D:\landslide\landslide_raster\LSM_stack_new_10m.tif"
mask_path = r"D:\landslide\landslide_raster\landslide_mask_10m.tif"

# Read data
with rasterio.open(stack_path) as src:
    stack = src.read()  # shape: (bands, rows, cols)

with rasterio.open(mask_path) as src:
    mask = src.read(1)

bands, rows, cols = stack.shape

# Reshape stack
stack_reshaped = stack.reshape(bands, -1).T   # shape: (pixels, bands)
mask_flat = mask.flatten()

# Landslide pixels
ls_idx = np.where(mask_flat == 1)[0]

# Non-landslide pixels
nls_idx = np.where(mask_flat == 0)[0]

# Randomly sample non-landslide equal to landslide
np.random.seed(42)
nls_sample = np.random.choice(nls_idx, size=len(ls_idx), replace=False)

# Combine indices
final_idx = np.concatenate([ls_idx, nls_sample])

# Extract features
X = stack_reshaped[final_idx]
y = mask_flat[final_idx]

In [3]:
import pandas as pd
# Create dataframe
df = pd.DataFrame(X)
df["label"] = y

print("Dataset shape:", df.shape)
print(df["label"].value_counts())

Dataset shape: (71846, 17)
label
1    35923
0    35923
Name: count, dtype: int64


In [4]:
feature_names = [
    "elevation",
    "slope",
    "aspect",
    "curvature",
    "TRI",
    "TWI",
    "rainfall",
    "dist_stream",
    "dist_road",
    "dist_fault",
    "NDVI",
    "BSI",
    "soil",
    "lithology",
    "LULC",
    "geomorphon"
]

df.columns = feature_names + ["label"]

print(df.head())

     elevation      slope      aspect  curvature        TRI        TWI  \
0  3964.060547  53.417248  308.530182  -0.000697  23.812599  10.389105   
1  3955.881348  53.340611  308.530182  -0.001078  23.698778  10.507830   
2  3947.371826  52.557346  311.633545  -0.001786  23.103390  10.388711   
3  3960.437500  52.179382  306.869904  -0.001116  22.586826   9.671568   
4  3952.341553  52.095924  308.530182  -0.001254  22.354893  10.597772   

      rainfall   dist_stream     dist_road    dist_fault      NDVI       BSI  \
0  1674.596313  27237.910156  29796.917969  89959.109375  0.100897  0.214427   
1  1674.721069  27247.183594  29806.853516  89969.101562  0.129019  0.175965   
2  1674.845825  27256.457031  29816.787109  89979.101562  0.132021  0.170064   
3  1674.510132  27232.380859  29788.134766  89948.968750  0.046843  0.273991   
4  1674.634888  27241.652344  29798.068359  89958.968750  0.082975  0.236869   

     soil  lithology  LULC  geomorphon  label  
0  3717.0        1.0  11.0

In [5]:
df.to_csv("New_LSM_Data.csv", index=False)

In [6]:
df.isna().sum()

elevation          0
slope              0
aspect             0
curvature          0
TRI                0
TWI            24269
rainfall           0
dist_stream        0
dist_road          0
dist_fault         0
NDVI               0
BSI                0
soil               0
lithology          0
LULC               0
geomorphon         0
label              0
dtype: int64

In [7]:
import numpy as np
df = df.replace(-9999, np.nan)

In [8]:
df.isna().sum()

elevation      19655
slope          19695
aspect         19851
curvature      19655
TRI            19599
TWI            24269
rainfall           0
dist_stream        0
dist_road          0
dist_fault         0
NDVI           19607
BSI            19607
soil               0
lithology          0
LULC               0
geomorphon     19655
label              0
dtype: int64

In [9]:
df = df.dropna()

In [10]:
df.label.value_counts()

label
1    32561
0    14923
Name: count, dtype: int64

In [11]:
df.to_csv("New_LSM_Data_Cleaned.csv", index=False)